# Machine Learning Pipeline - Scoring New Data

Imaginemos que un colega del área de negocio nos pide hacer predicciones sobre datos nuevos. Veremos dos formas de hacerlo.

---

## 1. IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import joblib

---

## 2. CARGAR DATOS NUEVOS

Para este ejemplo, usaremos una muestra de `data/test.csv` como si fueran datos completamente nuevos.

In [ ]:
# Cargar datos nuevos (o test.csv como ejemplo de unseen data)
data_new = pd.read_csv('data/test.csv')
print('Shape:', data_new.shape)
data_new.head()

# Eliminar columna ID si existe
data_new = data_new.drop('Id', axis=1, errors='ignore')

---

# 3. MÉTODO 1 — SCORING MANUAL (Python puro)

Este enfoque consiste en aplicar **cada transformación del feature engineering de manera manual**, con los valores aprendidos hardcodeados. Es muy propenso a errores, poco mantenible y difícil de escalar.

## a) Castear variables categóricas disfrazadas de numéricas

In [ ]:
# Supongamos que MSSubClass es numérica pero debe ser string
if 'MSSubClass' in data_new.columns:
    data_new['MSSubClass'] = data_new['MSSubClass'].astype(str)

## b) Imputar categóricas: fillna("Missing") y fillna(moda)

Hardcodeamos las listas y modas aprendidas en train:

In [ ]:
a_cat_high = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
a_cat_low = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtFinType2', 'BsmtExposure', 'BsmtFinType1', 'BsmtCond', 'BsmtQual', 'Electrical']
modas = {'GarageType': 'Attchd', 'GarageFinish': 'Unf', 'GarageQual': 'TA', 'GarageCond': 'TA',
        'BsmtFinType2': 'Unf', 'BsmtExposure': 'No', 'BsmtFinType1': 'Unf', 'BsmtCond': 'TA', 'BsmtQual': 'TA', 'Electrical': 'SBrkr'}

for col in a_cat_high:
    if col in data_new.columns:
        data_new[col] = data_new[col].fillna('Missing')
for col in a_cat_low:
    if col in data_new.columns:
        data_new[col] = data_new[col].fillna(modas[col])

## c) Imputar numéricas: indicador binario + fillna(media)

In [ ]:
a_num = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']
medias = {'LotFrontage': 69.305795,
          'GarageYrBlt': 1978.506164,
          'MasVnrArea': 103.685262}
for col in a_num:
    if col in data_new.columns:
        data_new[col + '_was_missing'] = data_new[col].isnull().astype(int)
        data_new[col] = data_new[col].fillna(medias[col])

## d) Variables temporales: calcular años transcurridos

In [ ]:
if {'YearBuilt','YearRemodAdd','GarageYrBlt','YrSold'}.issubset(data_new.columns):
    data_new['YearBuilt_elapsed']    = data_new['YrSold'] - data_new['YearBuilt']
    data_new['YearRemodAdd_elapsed'] = data_new['YrSold'] - data_new['YearRemodAdd']
    data_new['GarageYrBlt_elapsed']  = data_new['YrSold'] - data_new['GarageYrBlt']
    data_new = data_new.drop(['YearBuilt','YearRemodAdd','GarageYrBlt','YrSold'], axis=1, errors='ignore')

## e) Transformaciones: log, Yeo-Johnson (hardcodear lambda), binarizar ceros

Supongamos que las columnas altamente sesgadas con log son (como definido): 
`MiscVal, PoolArea, LotArea, 3SsnPorch, LowQualFinSF, KitchenAbvGr, BsmtFinSF2, ScreenPorch, BsmtHalfBath, EnclosedPorch, MasVnrArea, OpenPorchSF, LotFrontage, BsmtFinSF1, WoodDeckSF, TotalBsmtSF, MSSubClass, 1stFlrSF, GrLivArea, BsmtUnfSF, 2ndFlrSF`
Usamos log1p si positivos, y Yeo-Johnson con lambda hardcodeado para otras.

In [ ]:
skewed_cols = ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'MSSubClass', '1stFlrSF', 'GrLivArea', 'BsmtUnfSF', '2ndFlrSF']
# Hardcode: log1p si positivos según original, Yeo-Johnson para otros (simulamos)
yj_params = { # simulamos lambdas obtenidos previamente
    'KitchenAbvGr': (0.7,), 'BsmtFinSF2': (0.9,), # etc.
}
from scipy.stats import yeojohnson
for col in skewed_cols:
    if col in data_new.columns:
        if (data_new[col]>=0).all():
            data_new[col+'_log1p'] = np.log1p(data_new[col])
        elif col in yj_params:
            data_new[col+'_yj'] = yeojohnson(data_new[col], *yj_params[col])

## f) Binarización de sparse/ceros

In [ ]:
sparse_cols = ['MasVnrArea','BsmtFinSF2','2ndFlrSF','LowQualFinSF','BsmtFullBath','BsmtHalfBath','HalfBath','WoodDeckSF','EnclosedPorch','3SsnPorch','ScreenPorch','PoolArea','MiscVal']
for col in sparse_cols:
    if col in data_new.columns:
        data_new[col + '_iszero'] = (data_new[col]==0).astype(int)

## g) Mappings ordinales (hardcodeado calidad)

In [ ]:
ordinal_map = {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0}
ordinal_vars = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']
for col in ordinal_vars:
    if col in data_new.columns:
        data_new[col + '_num'] = data_new[col].map(ordinal_map).astype(float)

## h) Rare labels

Simulamos que tenemos las categorías frecuentes aprendidas:

In [ ]:
frequent_neighborhoods = [ 'NAmes', 'CollgCr', 'OldTown', 'Edwards', 'Somerst', 'Sawyer', 'NWAmes', 'Gilbert', 'NridgHt', 'SawyerW', 'BrkSide' ]
if 'Neighborhood' in data_new.columns:
    data_new['Neighborhood'] = data_new['Neighborhood'].apply(lambda x: x if x in frequent_neighborhoods else 'Rare')

## i) Encoding ordinal

Hardcodeamos que para nominales se usó OrdinalEncoder y que valores desconocidos van a -1.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
nominal_cols = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'Heating', 'CentralAir', 'Functional', 'GarageType', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
ord_encoder = joblib.load('proyecto_pipeline_machine_learning_muestra/ordinal_encoder.pkl')  # Debe haberse guardado antes
if set(nominal_cols) <= set(data_new.columns):
    nom_enc = pd.DataFrame(ord_encoder.transform(data_new[nominal_cols]), columns=[c+'_ord' for c in nominal_cols], index=data_new.index)
    data_new = pd.concat([data_new, nom_enc], axis=1)

## j) Verificar NaNs restantes

In [ ]:
remains_na = data_new.isnull().sum().sum()
if remains_na > 0:
    print(f'Hay {remains_na} NaN en data_new. Parcheando con 0 (NO recomendado).')
    data_new = data_new.fillna(0)

## k) Feature scaling: cargar y aplicar scaler

In [ ]:
scaler = joblib.load('proyecto_pipeline_machine_learning_muestra/scaler.pkl')
# Seleccionar columnas numéricas transformadas (usamos solo las seleccionadas luego)

## l) Seleccionar features: cargar selected_features

In [ ]:
selected_features = pd.read_csv('proyecto_pipeline_machine_learning_muestra/selected_features.csv').squeeze().tolist()
data_selected = data_new[selected_features]
data_selected = pd.DataFrame(scaler.transform(data_selected), columns=data_selected.columns)

## m) Cargar modelo y hacer predicción

In [ ]:
model = joblib.load('proyecto_pipeline_machine_learning_muestra/model.pkl')
preds_log = model.predict(data_selected)
preds = np.exp(preds_log)

## n) Visualizar predicciones

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(preds, bins=30, color='skyblue', edgecolor='k')
plt.title('Distribución de predicciones (scoring manual)')
plt.xlabel('Predicción SalePrice')
plt.ylabel('Frecuencia')
plt.show()

---

**PROBLEMAS del enfoque manual:**
- Muchísimo código repetitivo y pasos propensos a errores.
- Cada valor (medias, modas, frecuencias, parámetros de transformaciones) debe ser hardcodeado y sincronizado con el train.
- Se procesan variables incluso si el modelo no las usa.
- Si la data nueva tiene NaN inesperados, todo falla o se parchea mal.

---

# 4. MÉTODO 2 — SCORING CON PIPELINE (recomendado)

El Pipeline encapsula TODAS las transformaciones necesarias y aprendidas del set de entrenamiento. Así, sólo necesitas unas pocas líneas y es 100% reproducible y mantenible.

## a) Cargar datos nuevos LIMPIOS (sin transformar)

In [ ]:
data_new2 = pd.read_csv('data/test.csv').drop('Id', axis=1, errors='ignore')

## b) Castear variables categóricas disfrazadas (si corresponde)

In [ ]:
if 'MSSubClass' in data_new2.columns:
    data_new2['MSSubClass'] = data_new2['MSSubClass'].astype(str)

## c) Cargar pipeline

In [ ]:
pipeline = joblib.load('proyecto_pipeline_machine_learning_muestra/preprocessors.pkl')

## d) Transformar datos nuevos

In [ ]:
data_transformed = pipeline.transform(data_new2)

## e) Cargar features seleccionadas

In [ ]:
selected_features = pd.read_csv('proyecto_pipeline_machine_learning_muestra/selected_features.csv').squeeze().tolist()
# Si el pipeline retorna array, convertir a DataFrame y seleccionar columnas
if isinstance(data_transformed, np.ndarray):
    data_transformed = pd.DataFrame(data_transformed, columns=pipeline.get_feature_names_out())
data_selected2 = data_transformed[selected_features]

## f) Cargar modelo y predecir

In [ ]:
model = joblib.load('proyecto_pipeline_machine_learning_muestra/model.pkl')
preds_log2 = model.predict(data_selected2)
preds2 = np.exp(preds_log2)

## g) Visualización de predicciones

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(preds2, bins=30, color='seagreen', edgecolor='k')
plt.title('Distribución de predicciones (pipeline)')
plt.xlabel('Predicción SalePrice')
plt.ylabel('Frecuencia')
plt.show()

---

> **Observa:** Con solo unas pocas líneas, el pipeline replica TODAS las transformaciones. Es reproducible, mantenible y no necesitas hardcodear nada.

# 5. CONCLUSIONES

- El Pipeline encapsula todas las transformaciones aprendidas del set de entrenamiento (imputadores, codificadores, log/yeo-johnson, binarizadores, scalers, etc.).
- No hay riesgo de olvidar un paso, ni de hardcodear un valor incorrectamente, ni de aplicar una transformación inconsistente.
- Es la única forma recomendada (y mantenible) para scoring en producción o sobre datos nuevos.